# Objectives
Welcome to Lab 9 of Natural Language Processing. This notebook takes a look at language modeling by using an LSTM to learn next-character predicton. The code here is directly copied from [Raschka (2022)](https://sebastianraschka.com/books/machine-learning-with-pytorch-and-scikit-learn/) Chapter 15 "Modeling Sequential Data Using Recurrent Neural Networks" with minor modifications.

# Notebook Setup

## Packages
The **PyTorch framework documentation** is useful to get familiar with the various concepts and components, particularly:

- [Quickstart](https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html)
- [Datasets & DataLoaders](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html)
- [Build Model](https://docs.pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)

In [1]:
#| code-summary: 'Import packages'
# Data packages
import numpy as np

# ML/DL/NN packages
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.distributions.categorical import Categorical

## Moving Model and Data to GPU
It is generally faster to run on your GPU if available. This code checks for availability and subsequent code will move the model and data batches to the assigned device. 

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda') # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = torch.device('mps') # Apple GPU
else:
    device = torch.device('cpu') # Fallback CPU
print('Using device:', device)

Using device: cpu


# 1. Prepare Dataset
Go to https://www.gutenberg.org/ and find a favourite or interesting classic book:
- Find book page
- Under "Other formats and older devices", download the **Plain Text** file
- Move the text file to the same directory as this notebook
- Update the filename in the code cell below

In [3]:
## Reading and processing text
filename = 'birthmark.txt' # MODIFY FILENAME HERE

with open(filename, 'r', encoding="utf8") as fp:
    text=fp.read()
#start_indx = text.find('THE MYSTERIOUS ISLAND')
#end_indx = text.find('End of the Project Gutenberg')
#text = text[start_indx:end_indx]
char_set = set(text)
print('Total Length:', len(text))
print('Unique Characters:', len(char_set))

Total Length: 247125
Unique Characters: 91


## Build Character-Index "Vocabulary"

In [4]:
chars_sorted = sorted(char_set)
char2int = {ch:i for i,ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)
text_encoded = np.array(
    [char2int[ch] for ch in text],
    dtype=np.int32
)
print('Text encoded shape:', text_encoded.shape)
print(text[:15], '== Encoding ==>', text_encoded[:15])
print(text_encoded[15:21], '== Reverse ==>', ''.join(char_array[text_encoded[15:21]]))

Text encoded shape: (247125,)
The Project Gut == Encoding ==> [49 65 62  1 45 75 72 67 62 60 77  1 36 78 77]
[62 71 59 62 75 64] == Reverse ==> enberg


In [5]:
for ex in text_encoded[:5]:
    print('{} -> {}'.format(ex, char_array[ex]))

49 -> T
65 -> h
62 -> e
1 ->  
45 -> P


## Fixed Length Text Chunks

In [6]:
# Configure
seq_length = 40
num_chunks = 100000  # limit for training speed

chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size)]
text_chunks = text_chunks[:num_chunks]
len(text_chunks)

100000

## Setup PyTorch Dataset and Dataloader

In [7]:
# Configure data loading
batch_size = 64

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks
        
    def __len__(self):
        return len(self.text_chunks)

    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long() # source, target!!

seq_dataset = TextDataset(torch.tensor(text_chunks))

torch.manual_seed(1)
seq_dl = DataLoader(
    seq_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

C:\Users\Serenity\AppData\Local\Temp\ipykernel_28156\3780267394.py:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:255.)
  seq_dataset = TextDataset(torch.tensor(text_chunks))


## Check Source and Target Sequences

In [8]:
for i, (seq, target) in enumerate(seq_dataset):
    print(' Input (x): ', repr(''.join(char_array[seq])))
    print('Target (y): ', repr(''.join(char_array[target])))
    print()
    if i == 2:
        break

 Input (x):  'The Project Gutenberg eBook of Nathaniel'
Target (y):  'he Project Gutenberg eBook of Nathaniel '

 Input (x):  'he Project Gutenberg eBook of Nathaniel '
Target (y):  'e Project Gutenberg eBook of Nathaniel H'

 Input (x):  'e Project Gutenberg eBook of Nathaniel H'
Target (y):  ' Project Gutenberg eBook of Nathaniel Ha'



# 2. Define Model
You can modify the below parameters and monitor how it changes your output.

In [20]:
# Configure
vocab_size = len(char_array)
embed_dim = 256 # learned embedding dimensions
rnn_hidden_size = 256 # larger internal memory
num_layers = 2 # 2 or higher for deeper learning

We use the pre-made [nn.LSTM](https://docs.pytorch.org/docs/2.15/generated/torch.nn.LSTM.html) from PyTorch and set the parameters.

In [21]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim) # learns character embeddings
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)
        #self.softmax = nn.LogSoftmax(dim=1) # missing in textbook
        
    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        #out = self.softmax(out) # missing in textbook
        return out, hidden, cell

    def init_hidden(self, batch_size):
        hidden = torch.zeros(num_layers, batch_size, self.rnn_hidden_size, device=device) # added device
        cell = torch.zeros(num_layers, batch_size, self.rnn_hidden_size, device=device) # added device
        return hidden, cell

torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size).to(device)
model

RNN(
  (embedding): Embedding(91, 256)
  (rnn): LSTM(256, 256, num_layers=2, batch_first=True)
  (fc): Linear(in_features=256, out_features=91, bias=True)
)

# 3. Train Model

**Exercise:** Iteratively run a few hundred epochs at a time and observe your sample text (hopefully) improve its language modelling in terms of next-character prediction.

Run-time note:
- Wall time for LSTM, num_epochs=1001 (Apple GPU, 100k chunks): 2min 28s

In [22]:
%%time
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 1001 # upto 10000
torch.manual_seed(1)

for epoch in range(num_epochs):
    hidden, cell = model.init_hidden(batch_size)
    seq_batch, target_batch = next(iter(seq_dl))

    # Added
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)
    
    optimizer.zero_grad()
    loss = 0
    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])
    loss.backward()
    optimizer.step()
    loss = loss.item()/seq_length
    if epoch % 100 == 0:
        print(f'Epoch {epoch} loss: {loss:.4f}')

Epoch 0 loss: 4.4899
Epoch 100 loss: 2.3860
Epoch 200 loss: 2.1110
Epoch 300 loss: 1.9884
Epoch 400 loss: 1.8404
Epoch 500 loss: 1.7836
Epoch 600 loss: 1.7377
Epoch 700 loss: 1.6656
Epoch 800 loss: 1.5885
Epoch 900 loss: 1.6485
Epoch 1000 loss: 1.4655
CPU times: total: 1h 9min 16s
Wall time: 6min 40s


# 4. Evaluate Model
Define a function that takes in a `starting_str` and uses **auto-regression** to continuously add predicted characters to the input string until reaching the desired length at `len_generated_text`.

In [23]:
def sample(model, starting_str, len_generated_text=200, scale_factor=1.0):
    encoded_input = torch.tensor(
        [char2int[s] for s in starting_str],
        device=device # added
    )
    encoded_input = torch.reshape(
        encoded_input, (1, -1)
    )
    generated_str = starting_str
    
    model.eval()
    hidden, cell = model.init_hidden(1)
    for c in range(len(starting_str)-1):
        _, hidden, cell = model(encoded_input[:, c].view(1), hidden, cell)
    
    last_char = encoded_input[:, -1]
    for i in range(len_generated_text):
        logits, hidden, cell = model(last_char.view(1), hidden, cell)
        logits = torch.squeeze(logits, 0)
        scaled_logits = logits * scale_factor
        m = Categorical(logits=scaled_logits)
        last_char = m.sample()
        generated_str += str(char_array[last_char])

    return generated_str

In [25]:
torch.manual_seed(1)
print(sample(model, starting_str='The island'))

The islanded an old woods. Yet, it is crintwerest canderigup of the decrement, theark of a gleased glazed. Bray les; dy moring betwein the largely
carples, lew
cricceanceature
endosoomed her the umfected this o
